# 🫁 Pneumonia Detection from Chest X-Rays — End-to-End ML Notebook

**Course:** Machine Learning Cycle — Summative (image / non-tabular extension)
**Author:** Mahlet Tilahun
**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) — Paul Mooney (real medical imaging data, **not synthetic**)

This notebook covers the full ML lifecycle:
1. Data acquisition
2. Data processing & **detailed preprocessing**
3. Exploratory visualisations — **3 interpreted features**
4. Model creation (MobileNetV2 transfer learning + regularisation, early stopping)
5. Model training
6. Model testing / evaluation — **Accuracy, Loss, Precision, Recall, F1, AUC, confusion matrix**
7. Prediction function
8. Saving the model (`.keras` / `.h5`)
9. Retraining function & trigger

> The reusable logic lives in `src/` (`preprocessing.py`, `model.py`, `prediction.py`).
> This notebook imports and demonstrates it so nothing is duplicated.


## 0 · Setup

Run this from the **project root** with the dataset already downloaded into `data/`
(see `scripts/download_data.py`). Use a **Python 3.10/3.11** environment — TensorFlow 2.15
does not support Python 3.13.


In [ ]:
import sys, os, json
# Make the project's src/ importable when running from notebook/
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_style("whitegrid")

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 1 · Data Acquisition

The dataset is the Kaggle *Chest X-Ray Images (Pneumonia)* collection: 5,863 real X-ray JPEGs
split into `train/`, `test/`, `val/`, each with `NORMAL/` and `PNEUMONIA/` subfolders.

Download it once with:
```bash
python scripts/download_data.py     # automatic (needs a Kaggle token)
```
or manually (see README). The cell below just checks it is present.


In [ ]:
from src.config import TRAIN_DIR, TEST_DIR, VAL_DIR, CLASS_NAMES, IMG_SIZE

for split, d in [("train", TRAIN_DIR), ("test", TEST_DIR), ("val", VAL_DIR)]:
    for cls in CLASS_NAMES:
        p = d / cls
        n = len(list(p.glob("*"))) if p.exists() else 0
        print(f"{split:5s} / {cls:9s}: {n:5d} images")

## 2 · Data Processing & Detailed Preprocessing

Preprocessing steps applied to every image (all implemented in `src/preprocessing.py`):

| Step | What | Why |
|------|------|-----|
| **Resize** | all images → 150×150 | X-rays come in many resolutions; the model needs a fixed input |
| **Channel** | grayscale → RGB (3 ch) | MobileNetV2 was pretrained on 3-channel ImageNet |
| **Rescale** | pixel values ÷ 255 → [0,1] | stabilises gradient descent |
| **Augment (train only)** | rotation, shift, zoom, shear, h-flip | regularisation → less over-fitting on the small NORMAL class |
| **Class weights** | balanced weighting | dataset is ~3× more PNEUMONIA → prevents majority bias |

We build the generators and inspect one augmented batch.


In [ ]:
from src.preprocessing import build_generators, compute_class_weights

train_gen, val_gen, test_gen = build_generators()
class_weights = compute_class_weights(train_gen)
print("Class indices:", train_gen.class_indices)
print("Class weights:", class_weights)
print("Train batches:", len(train_gen), "| Val:", len(val_gen), "| Test:", len(test_gen))

In [ ]:
# Visualise one augmented training batch
images, labels = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(CLASS_NAMES[int(labels[i])])
    ax.axis("off")
plt.suptitle("Sample augmented & preprocessed training images (150×150, rescaled)")
plt.tight_layout(); plt.show()

## 3 · Exploratory Visualisations — 3 Interpreted Features

We interpret **three features** of the dataset and the story each tells.


In [ ]:
from src.preprocessing import dataset_summary
summary = dataset_summary(TRAIN_DIR, sample_per_class=120)
summary["class_counts"]

### Feature 1 — Class distribution (class balance)

In [ ]:
counts = summary["class_counts"]
plt.figure(figsize=(6,4))
plt.bar(counts.keys(), counts.values(), color=["#22c55e", "#ef4444"])
plt.title("Feature 1 — Class distribution in the training set")
plt.ylabel("number of images")
for i,(k,v) in enumerate(counts.items()):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

**Story:** The classes are **imbalanced** — pneumonia images are roughly three times as many
as normal ones. A naive model could reach high accuracy by always predicting PNEUMONIA.
We mitigate this with **class weights** and by tracking **recall & F1**, not just accuracy.

### Feature 2 — Mean pixel brightness per class (image intensity)

In [ ]:
bright = summary["brightness"]
means = {c: bright[c]["mean"] for c in bright}
plt.figure(figsize=(6,4))
plt.bar(means.keys(), means.values(), color=["#38bdf8", "#818cf8"])
plt.title("Feature 2 — Mean pixel intensity (0–255) per class")
plt.ylabel("mean brightness")
for i,(k,v) in enumerate(means.items()):
    plt.text(i, v, f"{v:.1f}", ha="center", va="bottom")
plt.show()

# Distribution view
plt.figure(figsize=(7,4))
for c in bright:
    sns.kdeplot(bright[c]["values"], label=c, fill=True, alpha=.3)
plt.title("Brightness distribution per class"); plt.xlabel("mean pixel intensity"); plt.legend(); plt.show()

**Story:** Pneumonia X-rays tend to contain **denser white opacities** (fluid/infection in the
lungs), which shifts their brightness distribution relative to normal lungs. This is precisely the
visual cue the CNN learns — the feature is genuinely informative, not noise.

### Feature 3 — Image dimensions (acquisition consistency)

In [ ]:
dims = pd.DataFrame(summary["dimensions"])
plt.figure(figsize=(7,5))
for c, col in zip(CLASS_NAMES, ["#22c55e", "#ef4444"]):
    sub = dims[dims["class"] == c]
    plt.scatter(sub["width"], sub["height"], s=12, alpha=.5, label=c, color=col)
plt.xlabel("width (px)"); plt.ylabel("height (px)")
plt.title("Feature 3 — Raw image dimensions before resizing"); plt.legend(); plt.show()
print(dims[["width","height"]].describe())

**Story:** The raw X-rays are captured at **many different resolutions and aspect ratios**.
Feeding these directly would make the model sensitive to acquisition hardware rather than pathology.
Our preprocessing **resizes every image to 150×150**, removing this variability so the model focuses
on lung content.

## 4 · Model Creation

We use **MobileNetV2 transfer learning** (ImageNet weights, frozen backbone) with a custom head.
Optimisation / regularisation techniques used (rubric "excellent" tier):
- **Pretrained model** (MobileNetV2)
- **Dropout (0.3)** and **L2 weight regularisation** in the head
- **Adam** optimiser with tuned learning rate
- **Data augmentation** (in the generator)
- **Early stopping** + **ReduceLROnPlateau** (next section)


In [ ]:
from src.model import build_model
model = build_model(model_type="mobilenet", learning_rate=1e-4)
model.summary()

## 5 · Model Training

`train_model()` wires in EarlyStopping (patience 4, restores best weights),
ReduceLROnPlateau, and ModelCheckpoint. Reduce `epochs` if running on CPU.


In [ ]:
from src.model import train_model
model, history = train_model(model, train_gen, val_gen, epochs=15, class_weights=class_weights)

In [ ]:
# Training curves
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

## 6 · Model Testing / Evaluation

We evaluate on the **held-out test set** with the full metric suite the rubric requires:
**Accuracy, Loss, Precision, Recall, F1, AUC**, plus the confusion matrix and a classification report.


In [ ]:
from src.model import evaluate_model
metrics = evaluate_model(model, test_gen)
print(json.dumps({k: metrics[k] for k in
      ["accuracy","precision","recall","f1_score","auc","loss","n_test_samples"]}, indent=2))

In [ ]:
import numpy as np
cm = np.array(metrics["confusion_matrix"])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (test set)")
plt.show()

print("Classification report:")
rep = pd.DataFrame(metrics["classification_report"]).T
rep

**Interpretation.** In a medical screening context **recall (sensitivity)** on the PNEUMONIA
class matters most — a missed pneumonia case (false negative) is more dangerous than a false alarm.
We report it explicitly alongside precision and F1. AUC summarises the ranking quality across all
thresholds.

## 7 · Prediction Function — predict a single X-ray

`src/prediction.py` exposes `predict()` used by both the notebook and the API.


In [ ]:
from src.prediction import predict, reset_model_cache
reset_model_cache()  # ensure we load the just-saved model

# Grab one real test image and predict it
sample = next((TEST_DIR / "PNEUMONIA").glob("*.jpeg"), None) or next((TEST_DIR / "PNEUMONIA").glob("*.jpg"))
result = predict(str(sample))
print("File:", sample.name)
print(json.dumps(result, indent=2))

plt.imshow(plt.imread(sample), cmap="gray")
plt.title(f"Prediction: {result['prediction']}  ({result['confidence']*100:.1f}%)")
plt.axis("off"); plt.show()

## 8 · Save the Model

Saved in both the native `.keras` format and legacy `.h5` (rubric-compatible),
into `models/`. This is the artifact the API loads in production.


In [ ]:
from src.model import save_model
save_model(model)
print("Saved:")
for f in Path("../models").glob("pneumonia_model.*"):
    print(" ", f.name, f"({f.stat().st_size/1e6:.1f} MB)")

## 9 · Retraining Function & Trigger

`src/model.retrain()` implements the retraining cycle used by the deployed **Retrain button**:
1. Merge user-uploaded images from `data/uploads/<class>/` into `data/train/<class>/`
2. Rebuild generators (now including the new data)
3. Train a fresh model → evaluate → save (overwriting the deployed model)

**Trigger:** in production this runs either (a) when a user clicks *Trigger Retraining* in the UI,
or (b) automatically once `RETRAIN_TRIGGER_THRESHOLD` new images have been uploaded (see `/upload`).

The cell below demonstrates the function (uses a few epochs for speed).


In [ ]:
from src.model import retrain
# Demo: retrain for a couple of epochs. In production this is called by the API.
# result = retrain(model_type="mobilenet", epochs=3)
# print(json.dumps(result["new_metrics"], indent=2))
print("retrain() is ready — trigger it from the deployed UI or uncomment above to run here.")

---
## Summary

- ✅ Real, non-tabular data (chest X-ray images)
- ✅ Detailed preprocessing (resize, RGB, rescale, augmentation, class weights)
- ✅ 3 interpreted feature visualisations
- ✅ Transfer-learning model with regularisation + early stopping
- ✅ Full evaluation metric suite (accuracy, loss, precision, recall, F1, AUC, confusion matrix)
- ✅ Single-image prediction function
- ✅ Saved `.keras` / `.h5` model
- ✅ Retraining function + trigger

The same `src/` modules power the **FastAPI + Docker deployment** and the **Locust flood test**
(see the project `README.md`).
